# Segmentación de clientes para CRM

**Objetivo:** identificar a los clientes interesados en nuevas colecciones, a los recurrentes y a los de mayor valor, y agruparlos en arquetipos para que el equipo de CRM se comunique de forma distinta con cada grupo.

## 1. Carga y vista general

In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent
DATA = RAIZ / "data"

ARCHIVOS = ["customers", "orders", "order_items", "products", "stores"]

faltantes = [nombre for nombre in ARCHIVOS if not (DATA / f"{nombre}.csv").exists()]
assert not faltantes, f"Faltan archivos en {DATA}: {faltantes}"

customers   = pd.read_csv(DATA / "customers.csv")
orders      = pd.read_csv(DATA / "orders.csv")
order_items = pd.read_csv(DATA / "order_items.csv")
products    = pd.read_csv(DATA / "products.csv")
stores      = pd.read_csv(DATA / "stores.csv")

tablas = {"customers": customers, "orders": orders, "order_items": order_items,
          "products": products, "stores": stores}

for nombre, df in tablas.items():
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")

customers: 3122 filas x 10 columnas
orders: 22676 filas x 6 columnas
order_items: 48293 filas x 5 columnas
products: 240 filas x 5 columnas
stores: 5 filas x 2 columnas


In [2]:
for nombre, df in tablas.items():
    print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 3122 entries, 0 to 3121
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   customer_id       3122 non-null   str  
 1   email             3122 non-null   str  
 2   telefono          3112 non-null   str  
 3   fecha_nacimiento  3030 non-null   str  
 4   ciudad            3122 non-null   str  
 5   fecha_registro    3122 non-null   str  
 6   canal_registro    3122 non-null   str  
 7   opt_in_email      3122 non-null   bool 
 8   opt_in_whatsapp   3122 non-null   bool 
 9   notas_crm         165 non-null    str  
dtypes: bool(2), str(8)
memory usage: 201.4 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 22676 entries, 0 to 22675
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   order_id     22676 non-null  str  
 1   customer_id  21876 non-null  str  
 2   fecha        22676 non-null  str  
 3   canal     

**Hallazgos de la estructura**

Se cargaron 5 tablas: 3.122 clientes, 22.676 órdenes, 48.293 ítems de órdenes, 240 productos y 5 tiendas.

1. costumers trae una columna adicional llamada notas_crm
2. Las columnas numéricas (cantidad, precio_unitario, descuento_pct, precio_lista) están bien tipadas como enteros. Que el tipo sea correcto no garantiza que los valores lo sean: se revisan rangos más adelante.
3. descuento_pct es entero lo que sugiere una escala de 0-100, hay que confirmarlo en la siguiente sección
4. las fechas son texto, deben convertirse un revisar si hay formatos mezclados


## 2. Calidad de datos
### 2.1 Duplicados y valores nulos

In [3]:
for nombre, df in tablas.items():
    print(f"\n{nombre}")
    print("  filas duplicadas completas:", df.duplicated().sum())
    print("  nulos por columna:")
    print(df.isna().sum()[df.isna().sum() > 0].to_string())

ids = {"customers": "customer_id", "orders": "order_id",
       "products": "sku", "stores": "tienda_id"}

print("\nIDs repetidos (deberían ser únicos):")
for nombre, col in ids.items():
    print(f"  {nombre}.{col}: {tablas[nombre][col].duplicated().sum()}")


customers
  filas duplicadas completas: 0
  nulos por columna:
telefono              10
fecha_nacimiento      92
notas_crm           2957

orders
  filas duplicadas completas: 0
  nulos por columna:
customer_id      800
tienda_id      12143

order_items
  filas duplicadas completas: 627
  nulos por columna:
Series([], )

products
  filas duplicadas completas: 0
  nulos por columna:
fecha_lanzamiento    24

stores
  filas duplicadas completas: 0
  nulos por columna:
Series([], )

IDs repetidos (deberían ser únicos):
  customers.customer_id: 0
  orders.order_id: 0
  products.sku: 0
  stores.tienda_id: 0


**Hallazgos de duplicados y nulos**

| Tabla | Hallazgo | Hipótesis / impacto |
|---|---|---|
| customers | telefono (10) y fecha_nacimiento (92) nulos | Bajo impacto: no se usan para responder las tres preguntas, sin embargo sirven para revisar la integridad de otras columnas |
| customers | notas_crm vacía en 2.957 de 3.122 clientes (95%) | Solo 165 clientes tienen notas sin embargo algunas de las notas nos dan información que puede aportar de manera significativa al analisis |
| orders | 800 ordenes sin customer_id | Son ventas reales (tienen order_id), pero no se pueden atribuir a un cliente. Cuentan en el total de ventas, no en el análisis por cliente. |
| orders | 12.143 órdenes sin tienda_id | Probablemente son las órdenes online, que no tienen tienda física. |
| order_items | 627 filas duplicadas completas | Posible doble carga. Si no se corrige, infla las ventas y el valor de algunos clientes. |
| products | 24 productos sin fecha_lanzamiento | Verificado: todos son BASICOS, que no tienen temporada. No es un error. |


No hay IDs repetidos en ninguna tabla. Sin embargo, un ID único no garantiza que cada cliente sea una persona distinta: se revisa por email tambien

### 2.2 Plan de verificación

### Hallazgos iniciales y plan de verificación

| Código | Tabla | Análisis | Qué se revisa | Acción si se confirma |
|---|---|---|---|---|
| C1 | customers | Duplicados por email | Mismo email (normalizado) con distinto customer_id | Unificar al cliente más antiguo y reasignar sus órdenes |
| C2 | customers | Formato de email | Emails sin estructura válida | Marcar como no contactable por email |
| C3 | customers | Consistencia de texto | ciudad y canal_registro escritos de varias formas | Normalizar (minúsculas, espacios, sinónimos) |
| C4 | customers | Fechas de cliente | Formatos mezclados, registro futuro, edad < 15 o > 100 | Convertir; los valores imposibles pasan a nulo |
| C5 | customers | Consentimiento vs. contacto | Opt-in activo sin teléfono o con email inválido | Marcar como no activable en ese canal |
| C6 | customers | Contenido de notas_crm | Etiquetas útiles vs. textos con forma de instrucción | Etiquetas como contexto; los lineamientos se validan con el negocio antes de usarlos |
| O1 | orders | Consistencia de texto | canal y estado | Normalizar si hay variantes |
| O2 | orders | Canal vs. tienda | Tienda sin tienda_id u online con tienda_id | Corregir o documentar |
| O3 | orders | Integridad referencial | customer_id que no existe en customers, tienda_id que no existe en stores | Excluir del análisis por cliente; se mantienen en el total |
| O4 | orders | Fechas de orden | Formatos, rango, fechas futuras, órdenes antes del registro | Convertir; documentar los casos imposibles |
| O5 | orders | Distribución de estados | % de completadas, devueltas y canceladas | Solo las completadas cuentan como venta |
| O6 | orders | Ventas por tienda (descriptivo) | Participación de cada tienda y del canal online | Contexto de negocio |
| I1 | order_items | Duplicados exactos | 627 filas idénticas en todas las columnas | Pendiente de confirmar con el negocio; se calcula el impacto con y sin ellas |
| I2 | order_items | Rangos numéricos | cantidad ≤ 0, precio_unitario ≤ 0, descuentos irreales | Excluir o corregir según el caso |
| I3 | order_items | Integridad referencial | order_id que no existe en orders, sku que no existe en products | Excluir los ítems huérfanos |
| I4 | order_items | Precio vs. precio de lista | Precios muy alejados de precio_lista | Revisar si es un error de escala |
| P1 | products | Fechas de lanzamiento nulas | 24 productos sin fecha | Verificado: todos son BASICOS, sin temporada. No es un error. |
| P2 | products | Consistencia de colección | Nombre de colección, categoría y fecha de lanzamiento coherentes con la temporada | Normalizar y documentar las incoherencias |
| X1 | entre tablas | Órdenes sin ítems | Órdenes que no tienen ninguna línea en order_items | Documentar; no generan venta |

### 2.3 Verificación y limpieza

In [4]:
customers_clean   = customers.copy()
orders_clean      = orders.copy()
order_items_clean = order_items.copy()
products_clean    = products.copy()

registro = []

def registrar(codigo, tabla, problema, filas, accion):
    """Anota un hallazgo de calidad para la tabla resumen final."""
    registro.append({"codigo": codigo, "tabla": tabla, "problema": problema,
                     "filas_afectadas": filas, "accion": accion})

#### C1. Duplicados por email

In [5]:
email_norm = customers_clean["email"].str.strip().str.lower()

print("Repetidos tal cual:", customers_clean["email"].duplicated().sum())
print("Repetidos al normalizar:", email_norm.duplicated().sum())

# ¿Cuántas personas tienen más de un registro?
conteo = email_norm.value_counts()
print("Emails con más de un customer_id:", (conteo > 1).sum())
print("Máximo de registros para un mismo email:", conteo.max())

Repetidos tal cual: 0
Repetidos al normalizar: 116
Emails con más de un customer_id: 116
Máximo de registros para un mismo email: 2


In [6]:
dup_mask = email_norm.duplicated(keep=False)   # marca AMBOS registros de cada par
pares = customers_clean[dup_mask].assign(email_norm=email_norm[dup_mask])

# ¿Cómo se registró cada par? (ej. "online + tienda")
print(pares.groupby("email_norm")["canal_registro"]
           .apply(lambda s: " + ".join(sorted(s))).value_counts())

# ¿Los dos registros tienen compras?
ids_dup = pares["customer_id"]
print("\nIDs duplicados con órdenes:", ids_dup.isin(orders["customer_id"]).sum(), "de", len(ids_dup))

canal_registro
online + tienda    116
Name: count, dtype: int64

IDs duplicados con órdenes: 175 de 232


In [7]:
ordenes_afectadas = orders["customer_id"].isin(ids_dup).sum()
ambos = (pares.assign(compro=pares["customer_id"].isin(orders["customer_id"]))
              .groupby("email_norm")["compro"].all().sum())
print("Órdenes de clientes duplicados:", ordenes_afectadas)
print("Pares donde ambos registros tienen compras:", ambos)

Órdenes de clientes duplicados: 722
Pares donde ambos registros tienen compras: 65


In [8]:
cc = customers_clean.assign(
    email_norm=email_norm,
    fecha_registro_dt=pd.to_datetime(customers_clean["fecha_registro"], format="mixed", errors="coerce"),
).sort_values(["email_norm", "fecha_registro_dt"])          # de más antiguo a más reciente

# Datos que se calculan sobre el par completo
cc["omnicanal"] = cc.groupby("email_norm")["canal_registro"].transform("nunique") > 1
cc[["opt_in_email", "opt_in_whatsapp"]] = (
    cc.groupby("email_norm")[["opt_in_email", "opt_in_whatsapp"]].transform("all")
)

# Identidad: el registro más antiguo (ID, fecha y canal de registro)
cc["id_maestro"] = cc.groupby("email_norm")["customer_id"].transform("first")
mapa_ids = dict(zip(cc["customer_id"], cc["id_maestro"]))
maestros = cc[cc["customer_id"] == cc["id_maestro"]].set_index("email_norm")

# Contacto: el dato más reciente disponible (last ignora los nulos)
campos_contacto = ["telefono", "fecha_nacimiento", "ciudad", "notas_crm"]
maestros.update(cc.groupby("email_norm")[campos_contacto].last())
maestros["email"] = maestros.index                          # email ya normalizado

customers_clean = maestros.reset_index(drop=True).drop(columns=["id_maestro", "fecha_registro_dt"])
orders_clean["customer_id"] = orders_clean["customer_id"].map(mapa_ids).fillna(orders_clean["customer_id"])

registrar("C1", "customers", "Misma persona registrada en tienda y online con dos customer_id",
          116, "Se conserva el registro más antiguo y se toman los datos de contacto más recientes; "
               "las órdenes del duplicado se reasignan")

print("Clientes después de unificar:", len(customers_clean))

Clientes después de unificar: 3006


In [9]:
ids_eliminados = set(ids_dup) - set(customers_clean["customer_id"])
print("Clientes:", len(customers), "->", len(customers_clean))
print("IDs eliminados:", len(ids_eliminados))
print("Órdenes que aún apuntan a un ID eliminado:", orders_clean["customer_id"].isin(ids_eliminados).sum())
print("Total de órdenes sin cambio:", len(orders) == len(orders_clean))

Clientes: 3122 -> 3006
IDs eliminados: 116
Órdenes que aún apuntan a un ID eliminado: 0
Total de órdenes sin cambio: True


**Problema:** 116 personas están registradas dos veces, con el mismo email escrito con distinta capitalización o espacios y dos customer_id distintos. En los 116 casos un registro es de tienda y el otro online: los canales no comparten un perfil único de cliente.

**Tratamiento:** se normalizó el email (minúsculas y sin espacios), se conservó el registro más antiguo de cada persona y se le reasignaron las X órdenes de su duplicado. Las órdenes no se eliminan; solo cambian de dueño. El consentimiento de contacto queda activo solo si estaba activo en ambos registros. Resultado: 3.006 clientes, en línea con los aproximados 3.000 que describe el caso.

**Impacto si no se corrige:** en Y casos ambos registros tienen compras, así que el historial de esas personas queda partido en dos perfiles. Se verían menos recurrentes y de menor valor de lo que realmente son, lo que afecta directamente las preguntas 1, 2 y 3.

#### C2. Formato de email

El email es el canal principal para las campañas de CRM. Un email mal formado no se puede contactar, aunque el cliente haya dado su consentimiento. Se revisa que cada email tenga la estructura mínima usuario@dominio.extensión.

In [10]:
patron = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
customers_clean["email_valido"] = customers_clean["email"].str.match(patron)

invalidos = customers_clean.loc[~customers_clean["email_valido"], "email"]
print("Emails inválidos:", len(invalidos), f"({len(invalidos) / len(customers_clean):.1%})")

# ¿Qué tipo de error tienen? (sin mostrar los emails)
print("Sin @:", (~invalidos.str.contains("@")).sum())
print("Más de una @:", (invalidos.str.count("@") > 1).sum())
print("Sin punto en el dominio:", (invalidos.str.contains("@") & ~invalidos.str.split("@").str[-1].str.contains(r"\.")).sum())
print("Con espacios internos:", invalidos.str.contains(r"\s").sum())

print("\nDe los inválidos, cuántos tienen opt-in de email activo:",
      customers_clean.loc[~customers_clean["email_valido"], "opt_in_email"].sum())

Emails inválidos: 8 (0.3%)
Sin @: 2
Más de una @: 2
Sin punto en el dominio: 4
Con espacios internos: 0

De los inválidos, cuántos tienen opt-in de email activo: 8


In [11]:
registrar("C2", "customers", "Email con formato inválido (sin @, doble @ o sin dominio)", 8,
          "No se corrige para no adivinar el contacto; se marca email_valido = False")

**Problema:** 8 clientes (0,3%) tienen un email inválido: 2 sin @, 2 con más de una @ y 4 sin punto en el dominio. Los 8 tienen activo el consentimiento para recibir emails.

**Tratamiento:** no se corrigen automáticamente. Adivinar la dirección correcta podría enviar comunicaciones a otra persona, lo que es un riesgo de privacidad. Se conservan en el análisis (sus compras son válidas) y se marcan como no contactables por email.

**Impacto si no se corrige:** campañas con rebote y una base contactable inflada. Por su bajo volumen no afecta las conclusiones del análisis.

**Recomendación:** validar el formato del email en el momento del registro y solicitar la actualización a estos 8 clientes en su próxima compra.

#### C3. Consistencia de texto

ciudad y canal_registro se usan para agrupar clientes. Si un mismo valor está escrito de varias formas (mayúsculas, tildes, abreviaturas), un grupo se parte en varios y los conteos y ventas por ciudad o canal salen incorrectos.

In [12]:
for col in ["ciudad", "canal_registro"]:
    print(f"\n{col}:")
    print(customers_clean[col].value_counts(dropna=False).to_string())


ciudad:
ciudad
León                411
Monterrey           407
Querétaro           384
Guadalajara         371
Tijuana             365
Puebla              360
Mérida              358
Ciudad de México    350

canal_registro:
canal_registro
online    1634
tienda    1372


**Resultado:** sin hallazgos. Las 8 ciudades y los 2 canales de registro están escritos de forma consistente. Los clientes se reparten de forma similar entre ciudades (entre 350 y 411 cada una) y entre canales (1.634 online y 1.372 en tienda).

#### C4. Fechas de cliente

La fecha de registro define la antigüedad del cliente y la de nacimiento su edad. Si hay formatos mezclados, pandas puede leer mal las fechas (por ejemplo, confundir día y mes) o no leerlas. Primero se identifican los formatos presentes, reemplazando cada dígito por 9 para ver la forma de la fecha sin exponer el dato.

In [13]:
for col in ["fecha_registro", "fecha_nacimiento"]:
    formatos = customers_clean[col].str.replace(r"\d", "9", regex=True)
    print(f"\nFormatos de {col}:")
    print(formatos.value_counts(dropna=False).to_string())


Formatos de fecha_registro:
fecha_registro
9999-99-99    3006

Formatos de fecha_nacimiento:
fecha_nacimiento
9999-99-99    2923
NaN             83


Todas las fechas tienen la forma AAAA-XX-XX, sin formatos mezclados. Falta confirmar si es año-mes-día o año-día-mes: si la parte del medio supera 12 en algún caso, no puede ser un mes. Luego se convierten de texto a fecha con el formato explícito y se revisa que los valores tengan sentido.

In [14]:
for col in ["fecha_registro", "fecha_nacimiento"]:
    partes = customers_clean[col].str.split("-", expand=True).astype(float)
    print(f"{col}: parte del medio máx = {partes[1].max():.0f}, parte final máx = {partes[2].max():.0f}")

for col in ["fecha_registro", "fecha_nacimiento"]:
    convertida = pd.to_datetime(customers_clean[col], format="%Y-%m-%d", errors="coerce")
    no_convertidas = convertida.isna().sum() - customers_clean[col].isna().sum()
    print(f"\n{col}: {no_convertidas} fechas inválidas | rango {convertida.min().date()} a {convertida.max().date()}")
    customers_clean[col] = convertida

hoy = pd.Timestamp("2026-10-14")
edad = (hoy - customers_clean["fecha_nacimiento"]).dt.days / 365.25
print("\nRegistros en el futuro:", (customers_clean["fecha_registro"] > hoy).sum())
print("Edad < 15:", (edad < 15).sum(), "| Edad > 100:", (edad > 100).sum())
print("Registrado antes de nacer:", (customers_clean["fecha_registro"] < customers_clean["fecha_nacimiento"]).sum())

fecha_registro: parte del medio máx = 12, parte final máx = 31
fecha_nacimiento: parte del medio máx = 12, parte final máx = 28

fecha_registro: 0 fechas inválidas | rango 2022-01-01 a 2026-06-30

fecha_nacimiento: 0 fechas inválidas | rango 1900-01-01 a 2031-05-04

Registros en el futuro: 0
Edad < 15: 8 | Edad > 100: 34
Registrado antes de nacer: 8


La parte del medio nunca supera 12: el formato es año-mes-día, sin ambigüedad. Todas las fechas se convierten correctamente y no hay registros futuros. Sin embargo, la fecha de nacimiento va de 1900-01-01 a 2031-05-04: hay 34 clientes con más de 100 años y 8 con fecha de nacimiento futura.

In [15]:
print("Exactamente 1900-01-01:", (customers_clean["fecha_nacimiento"] == "1900-01-01").sum())

imposible = (edad > 100) | (edad < 15)
customers_clean.loc[imposible, "fecha_nacimiento"] = pd.NaT

registrar("C4", "customers", "Fecha de nacimiento imposible (valor por defecto 1900-01-01 o fecha futura)",
          int(imposible.sum()), "Se reemplaza por nulo; no se usa la edad en esos clientes")

print("Fechas de nacimiento anuladas:", imposible.sum())

Exactamente 1900-01-01: 34
Fechas de nacimiento anuladas: 42


**Resultado:** todas las fechas usan el formato año-mes-día y se convirtieron de texto a tipo fecha. No hay fechas de registro futuras; los registros van de 2022-01-01 a 2026-06-30.

**Problema:** 42 fechas de nacimiento son imposibles: 34 con edad mayor de 100 años (principalmente 1900-01-01, un valor por defecto del sistema) y 8 en el futuro (hasta 2031).

**Tratamiento:** se reemplazan por nulo. Inventar una edad metería datos falsos en el perfil de los arquetipos; es mejor un dato faltante que uno incorrecto.

#### C5. Consentimiento vs. datos de contacto

Para que una campaña llegue, el cliente necesita el consentimiento activo y un dato de contacto válido en ese canal. Se revisan los clientes con consentimiento pero sin un contacto utilizable, y el formato de los teléfonos (en México un celular tiene 10 dígitos, o 12 con el prefijo 52).

In [17]:
c = customers_clean
tel_digitos = c["telefono"].str.replace(r"\D", "", regex=True)

print("Opt-in WhatsApp sin teléfono:", (c["opt_in_whatsapp"] & c["telefono"].isna()).sum())
print("Opt-in email con email inválido:", (c["opt_in_email"] & ~c["email_valido"]).sum())

print("\nLongitud de los teléfonos (solo dígitos):")
print(tel_digitos.str.len().value_counts(dropna=False).sort_index().to_string())

Opt-in WhatsApp sin teléfono: 8
Opt-in email con email inválido: 8

Longitud de los teléfonos (solo dígitos):
telefono
2.0        4
4.0        4
10.0      48
12.0    2940
NaN       10


La mayoría de teléfonos tiene 12 dígitos (prefijo 52 + 10 dígitos). Hay 48 con 10 dígitos, válidos pero sin prefijo, y 8 con 2 o 4 dígitos, que no son un teléfono válido. Además, 8 clientes tienen opt-in de WhatsApp sin teléfono y 8 tienen opt-in de email con un email inválido (los identificados en C2).

In [18]:
# Estandarizar al formato 52 + 10 dígitos
tel = customers_clean["telefono"].str.replace(r"\D", "", regex=True)
tel = tel.where(tel.str.len() != 10, "52" + tel)            # a los de 10 dígitos se les agrega el prefijo
valido = tel.str.len().eq(12) & tel.str.startswith("52")

customers_clean["telefono"] = tel.where(valido)             # los inválidos quedan como nulo
customers_clean["telefono_valido"] = valido.fillna(False)

# Contactable = consentimiento + dato válido en ese canal
customers_clean["contactable_email"]    = customers_clean["opt_in_email"] & customers_clean["email_valido"]
customers_clean["contactable_whatsapp"] = customers_clean["opt_in_whatsapp"] & customers_clean["telefono_valido"]

registrar("C5", "customers", "Teléfono sin prefijo de país (10 dígitos)", 48,
          "Se estandariza agregando el prefijo 52")
registrar("C5", "customers", "Teléfono incompleto (2 o 4 dígitos)", 8,
          "Se reemplaza por nulo y se marca como no contactable por WhatsApp")

print("Opt-in WhatsApp no contactables:", (customers_clean["opt_in_whatsapp"] & ~customers_clean["telefono_valido"]).sum())
print("Opt-in email no contactables:", (customers_clean["opt_in_email"] & ~customers_clean["email_valido"]).sum())

Opt-in WhatsApp no contactables: 16
Opt-in email no contactables: 8


**Problema:** 48 teléfonos no tenían prefijo de país y 8 estaban incompletos. Además, algunos clientes tienen consentimiento activo sin un dato de contacto válido en ese canal.

**Tratamiento:** los teléfonos de 10 dígitos se estandarizaron con el prefijo 52; los incompletos se reemplazaron por nulo. No se modificó el consentimiento del cliente: se crearon las columnas contactable_email y contactable_whatsapp, que combinan consentimiento y dato válido.

**Impacto si no se corrige:** se sobrestimaría la base contactable y las campañas intentarían envíos que fallan. Con formatos de teléfono distintos, además, la herramienta de envío podría rechazar números válidos.